# 43 저장 답변 원문 재검토와 조건 라우팅 오프라인 검사
운영 코드·OCR·색인·이전 결과는 읽기 전용입니다. API와 모델을 새로 호출하지 않습니다.
144조건/97고유패킷의 핵심요구를 검토한 사후 진단입니다. 새 질문의 독립 확증·사람 블라인드 평가·전체 부수 문장 정확도 검증이 아닙니다.
helper는 동결 입력·원문검토 연결과 기존 파일 보존,반례 검증 책임을 가지며 이 노트북은 저장 결과를 표시합니다. 기존 유료 실험 runner를 다시 실행하지 않습니다.


In [1]:
from pathlib import Path
import json
ROOT=Path.cwd()
if ROOT.name=="notebooks":ROOT=ROOT.parent
DATA=ROOT/"notebooks/data/43_offline_validation"
def read(name):return json.loads((DATA/name).read_text())
print(json.dumps(read("protocol.json"),ensure_ascii=False,indent=2))


{
  "answer_conditions": 144,
  "diagnostic_questions": 4,
  "independent_holdout": false,
  "modes": [
    "U",
    "S_THEN_U",
    "S_PLUS_U"
  ],
  "new_api_requests": 0,
  "positive_questions": 44,
  "profiles": [
    "card_page_section_benefit"
  ],
  "review_contract_sha256": "fc58918d6563c1ef72b0b777b8dfe95d4ad830149d1c6d72d9360ee448bde53d",
  "sha256": "bec6ad35928a20a126b0a7447f0277d137429af19e6d72f8422abe7d56ba7bfb",
  "sources": {
    "notebooks/30_build_structured_db.py": "2254f3ce955e4c171e395229d80ee868a25565609b4bde83e41934a9d43da8a7",
    "notebooks/32_search_experiment.py": "8f67730989a0432c2dde8ec45b68a23a834a340b64fe4ce13e94c6ef3408371b",
    "notebooks/42_expanded_query_evaluation.py": "cd22600ae03da01bfc622bc3643c3d9bcd422dce46eb4230aea95b574fb58c98",
    "notebooks/data/30_structured_benefit_db/documents.jsonl": "fdf855a8d3b11103f776f913b010860c6b9c5e53af5b63a61738683c3d678eac",
    "notebooks/data/30_structured_benefit_db/facts.jsonl": "bdd116338687101a0362c70383

## 저장 답변 원문 판정
Core request satisfaction(핵심요구 충족):양성44질문 중 원질문 대상·행동·수치·AND/OR를 소유 근거로 충족한 PASS 수. 높을수록 좋고 다른 판정은0. 예:한 카드의GS25+넷플릭스 각각10% 이상 할인 근거를 설명. 목록 밖 유효 카드도 인정하며 모든 부수 문장의 정확도/미래 성공률은 아니다.
PARTIAL은일부 미확정,FAIL은미충족,UNCERTAIN은관계불명확으로 통과에 합산하지 않는다. 예:네이버페이 조건부 혜택만으로 네이버쇼핑 혜택을 확정 못하면PARTIAL.
Wrong core recommendation(잘못된 핵심추천):같은 양성질문에서 다른 추천의 대상·행동·율 오류가 있는 답변 수.0이 방어 기준,오류증가시 우세 판단 금지. 예:CU할인을적립으로주장. OCR 자체 오류/모든 부수 문장 검증은 아니다. 이전자동점수는42의 문자열/부분정답 통과 수이며 변경하지 않는다.


In [2]:
import pandas as pd
s=read("answer_review_summary.json")
display(pd.DataFrame(s["metrics"]))
print(json.dumps(s["paired"],ensure_ascii=False,indent=2))
print("대조:",s["diagnostics"])


,FAIL,PARTIAL,PASS,UNCERTAIN,group,mode,n,old_automatic_complete,wrong_core_recommendation
0,2,1,41,0,all,U,44,30,0
1,0,0,44,0,all,S_THEN_U,44,31,0
2,0,0,44,0,all,S_PLUS_U,44,33,0
3,1,1,14,0,numeric,U,16,12,0
4,0,0,16,0,numeric,S_THEN_U,16,12,0
5,0,0,16,0,numeric,S_PLUS_U,16,14,0
6,0,0,16,0,proper_noun,U,16,16,0
7,0,0,16,0,proper_noun,S_THEN_U,16,16,0
8,0,0,16,0,proper_noun,S_PLUS_U,16,16,0
9,1,0,11,0,semantic,U,12,2,0


[
  {
    "baseline": "U",
    "candidate": "S_THEN_U",
    "losses": [],
    "ties": [
      "E01",
      "E03",
      "E04",
      "E06",
      "E07",
      "E08",
      "E09",
      "E10",
      "E11",
      "E12",
      "E13",
      "E14",
      "E15",
      "E16",
      "E17",
      "E18",
      "E19",
      "E20",
      "E21",
      "E22",
      "E23",
      "E24",
      "E25",
      "E26",
      "E27",
      "E28",
      "E29",
      "E30",
      "E31",
      "E32",
      "E33",
      "E34",
      "E35",
      "E36",
      "E37",
      "E39",
      "E40",
      "E41",
      "E42",
      "E43",
      "E44"
    ],
    "wins": [
      "E02",
      "E05",
      "E38"
    ],
    "wrong_core_error_increase": false
  },
  {
    "baseline": "S_THEN_U",
    "candidate": "S_PLUS_U",
    "losses": [],
    "ties": [
      "E01",
      "E02",
      "E03",
      "E04",
      "E05",
      "E06",
      "E07",
      "E08",
      "E09",
      "E10",
      "E11",
      "E12",
      "E13",
      "E

## 실제 경로 재생과 통제 반례
Replay match는실제48계획의selection/fallback이 두프로필 저장96행과 일치한질문 수.48/48이 통과,의미추출 정확도는아니다.
Fixture match는실행전동결한28합성조건의카드/복귀/범위 기대값일치.28/28이 통과지만위험한현재동작도 관측기대에포함돼 안전성보장은아니다. 예:월한도요구에서 일한도카드는제외.
Lost correct cards/Admitted incorrect cards는합성oracle와 주입계획의차집합.낮을수록좋고실제오류빈도가아니다. 예:할인을적립으로잘못추출해할인카드가제외됨.


In [3]:
r=read("routing_results.json")
print("실제48 일치:",r["actual_replay_matches"],"통제28 일치:",r["controlled_expectation_matches"])
print("실제 복귀:",sum(v["fallback"] for v in r["actual_records"]))
display(pd.DataFrame([{k:v for k,v in x.items() if k not in ["normalized_plan","selection"]} for x in r["controlled_records"]]))

print("실제 검색 범위:",read("scope_audit.json")["checked_search_rows"],"행 일치; 범위지정",read("scope_audit.json")["actual_scoped_rows"],"행")


실제48 일치: 48 통제28 일치: 28
실제 복귀: 18


,case,correct_cards_lost,effective_scope,eligible_cards,fallback,incorrect_cards_admitted,matches_frozen_expectation,nonempty_selection_with_no_question_semantic_validation,parse_failure,parse_gate,risk_kind,unknown_cards
0,할인과적립구분,None,"[A, B]","[A, B]",False,None,True,True,None,ok,None,[]
1,적립만선택,None,[D],[D],False,None,True,True,None,ok,None,[]
2,같은카드AND,None,[A],[A],False,None,True,True,None,ok,None,[]
3,OR대안,None,"[A, B, C, I]","[A, B, C, I]",False,None,True,True,None,ok,None,[H]
4,월한도,None,[A],[A],False,None,True,True,None,ok,None,[]
5,일한도,None,[E],[E],False,None,True,True,None,ok,None,[]
6,전월실적경계와unknown,None,[A],[A],False,None,True,True,None,ok,None,[I]
7,최소율경계10포함,None,[A],[A],False,None,True,True,None,ok,None,[]
8,최소율경계11제외,None,[B],[B],False,None,True,True,None,ok,None,[]
9,월한도6000경계,None,[B],[B],False,None,True,True,None,ok,None,[]


실제 검색 범위: 192 행 일치; 범위지정 114 행


## 비용과 보존
신규 provider 요청·임베딩·재정렬 추론0. 실험 API 사용량 비용0달러이며 Codex 검토 모델 사용료/전체 작업시간과 실제 청구액은미측정. 아래시간/CPU/RSS는선택기재생프로세스만의측정이다. 모델/색인 구성과운영코드 변경0.


In [4]:
print(json.dumps(r["resources"],ensure_ascii=False,indent=2))
print(json.dumps(read("preservation.json"),ensure_ascii=False,indent=2))


{
  "cpu_seconds": 13.061142999999998,
  "gpu": "사용 안함",
  "new_embedding_vectors": 0,
  "new_usage_cost_usd": "0",
  "peak_rss_kib": 246964,
  "provider_requests": 0,
  "scope": "오프라인 재생 프로세스 측정. Codex 검토 시간/모델 사용료·실제 청구액은 별도이며 미측정",
  "wall_seconds": 15.39578653598437
}
{
  "changed": [],
  "checked_files": 25023,
  "passed": true
}


## 해석과 다음 단계
benefit 순차/융합은사후핵심요구44/44로동률이고 단독은41/44다. 기존기준후보benefit 순차를유지한다.
형식정상 의미오류는후보가남으면복귀되지않음,parser실패는명시범위를자동복원하지않음,빈AND그룹은전체후보를선택함을확인했다. 회귀통과가이위험의해소를뜻하지않는다.
운영API 연결·활성색인전환·실제LLM재생성·지연/비용측정은실행하지않았다.
